# actpatch demo — swapping an image with activation patching

This notebook demonstrates the core experiment behind `actpatch` on a real
Vision-Language Model, letting the model **generate a full response** (not just
one token) so you can watch its natural output before and after patching:

1. Show the two images (a **red apple** and a **cat**).
2. Let the model **describe** each image — it talks about an apple / a cat.
3. Run **activation patching**: transplant the cat image's activations into the
   apple run, leaving the text prompt untouched.
4. Let the model generate again on the patched apple run — it now describes a
   **cat**. The visual activations causally carried the "what is in the image"
   signal.

Run top-to-bottom. Needs a GPU and will download model weights on first run.
Default model is Qwen2.5-VL-3B; set `MODEL_ID` to an InternVL id to try that.

## 1. Setup

In [ ]:
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

from actpatch import (
    ActivationPatcher,
    CacheSpec,
    Component,
    PatchSpec,
    get_adapter,
    image_token_positions,
)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"   # or e.g. "OpenGVLab/InternVL3_5-1B-hf"
PROMPT = "Describe this image in one short sentence."
MAX_NEW_TOKENS = 40        # how many tokens to let the model generate
IMAGE_SIZE = 448           # square side (multiple of 28) so both images share one grid

# Locate the data/ folder whether the notebook runs from repo root or notebooks/.
DATA = Path("data") if Path("data").exists() else Path("../data")
APPLE, CAT = DATA / "red_apple.jpeg", DATA / "cat.jpeg"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print("device:", device)

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=dtype, device_map=device
).eval()

adapter = get_adapter(model)              # auto-detects the model family
patcher = ActivationPatcher(model, adapter)
print(type(model).__name__, "-", len(adapter.get_decoder_layers(model)), "decoder layers")

In [ ]:
# InternVL tiles images dynamically; disable it so every image is one fixed tile.
PROC_KWARGS = {"crop_to_patches": False} if "InternVL" in type(model).__name__ else {}


def build_inputs(image_path):
    """Build single-image chat inputs, resized to a fixed square."""
    image = Image.open(image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], return_tensors="pt", **PROC_KWARGS).to(device)


def generate_text(inputs, max_new_tokens=MAX_NEW_TOKENS):
    """Greedy-generate a full continuation and return the decoded new text."""
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 2. The two images

In [ ]:
import matplotlib.pyplot as plt  # pip install matplotlib if missing

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
for ax, path, title in [(axes[0], APPLE, "target: red_apple"), (axes[1], CAT, "source: cat")]:
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Baseline generation (no patching)

Let the model describe each image normally. The **apple** run is our *target*
(the one we will modify); the **cat** run is the *source* (the donor whose
activations we will copy).

In [ ]:
src = build_inputs(CAT)     # donor / source
tgt = build_inputs(APPLE)   # the run we modify / target

apple_baseline = generate_text(tgt)
cat_baseline = generate_text(src)
print("apple image:", apple_baseline)
print("cat   image:", cat_baseline)

## 4. Activation patching: transplant the cat into the apple run

We cache the cat image's activations at every image-token position across **all**
decoder layers (residual input + K + V), then keep those patches active while the
model **generates** on the apple inputs. The text prompt is never touched — only
the image activations are swapped.

Generation runs inside `patcher.patching(...)`: the patches apply during the
prefill (where the image tokens are present) and bake into the KV cache, so the
whole generated sentence reflects the swapped image. Using the model's own
`generate` means position-ids and the KV cache are handled natively.

In [ ]:
img_id = adapter.get_image_token_id(model)
src_pos = image_token_positions(src["input_ids"][0], img_id).tolist()
tgt_pos = image_token_positions(tgt["input_ids"][0], img_id).tolist()
assert len(src_pos) == len(tgt_pos), (
    f"image-token counts differ ({len(src_pos)} vs {len(tgt_pos)}); "
    "resize both images to the same square and disable tiling."
)
print(f"patching {len(tgt_pos)} image tokens across {len(adapter.get_decoder_layers(model))} layers")

layers = list(range(len(adapter.get_decoder_layers(model))))
comps = [Component.RESID_IN, Component.K, Component.V]

# 1) cache the source (cat) activations
cache = patcher.cache_source(src, CacheSpec.for_layers_tokens(layers, src_pos, comps))

# 2) re-key from source positions to the matching target positions
src_to_tgt = dict(zip(src_pos, tgt_pos))
for store in (cache.resid_in, cache.k_proj, cache.v_proj):
    remapped = {(L, src_to_tgt[t]): v for (L, t), v in store.items() if t in src_to_tgt}
    store.clear()
    store.update(remapped)

# 3) generate on the apple run with the cat patches active
patch = PatchSpec.for_layers_tokens(layers, tgt_pos, comps)
with patcher.patching(cache, patch):
    patched_text = generate_text(tgt)

print("apple PATCHED with cat:", patched_text)

## 5. Before vs after

In [ ]:
print("=" * 70)
print("apple image (baseline)   :", apple_baseline)
print("cat image (baseline)     :", cat_baseline)
print("apple image + CAT patch  :", patched_text)
print("=" * 70)
print("\nThe apple run's whole description flips to the cat once the image")
print("activations are patched in — the visual tokens carried the content.")

## 6. Peek: first-token distribution (optional)

If you want the raw next-token evidence rather than a generated sentence, look
at the top of the distribution at the first generated position. (Most striking
with a one-word prompt like `"... Answer with one word:"`.)

In [ ]:
def top_k(logits, k=8):
    tok = processor.tokenizer
    top = logits[0, -1].float().topk(k)
    return [(tok.decode([int(i)]).strip(), round(float(v), 2))
            for v, i in zip(top.values, top.indices)]

with torch.no_grad():
    apple_logits = model(**tgt).logits
patched_out = patcher.patched_forward(tgt, cache, patch, mode="online")
print("apple first-token, baseline:", top_k(apple_logits)[:6])
print("apple first-token, patched :", top_k(patched_out.logits)[:6])

## 7. Variations to try

- **Layer scan** — patch one layer at a time (`PatchSpec.for_layers_tokens([L], tgt_pos, comps)`)
  to find where the object identity becomes decisive.
- **Foreground only** — build a 2-D mask over `adapter.image_grid_shape(tgt, model)`
  with `rect_mask` / `mask_to_token_indices` to patch just the object (works
  cleanly on Qwen2.5-VL; InternVL reorders tokens, so prefer the full swap).
- **Offline mode** — `patcher.patched_forward(..., mode="offline", start_index=max(tgt_pos)+1)`
  writes K/V patches into the KV cache and decodes from after the image block.
- **Debug tracing** — `import actpatch; actpatch.enable_debug_logging()` prints
  every capture/patch so you can confirm the hooks fire.

See [`docs/GUIDE.md`](../docs/GUIDE.md) for full recipes and the
[`README`](../README.md) for the API reference and how to add a new model.